# Decision Tree
This notebook will develop the decision tree. It will include any research/sources I use to make the splits. The final version of the project will go into greater detail

In [9]:
import os
from pathlib import Path
import codecarbon.emissions_tracker as et

et.count_physical_cpus = lambda: 1

from codecarbon import EmissionsTracker

LOG_DIR = Path("data/carbon_logs")
LOG_DIR.mkdir(parents=True, exist_ok=True)

tracker = EmissionsTracker(
    project_name="decision_tree_builder",
    output_dir=str(LOG_DIR),
    output_file="decision_tree_builder.csv",
)
tracker.start()

[codecarbon INFO @ 15:40:40] Codecarbon configuration: allow_multiple_runs from CODECARBON_ALLOW_MULTIPLE_RUNS, cpu_physical_count from CODECARBON_CPU_PHYSICAL_COUNT
[codecarbon WARNING @ 15:40:40] Multiple instances of codecarbon are allowed to run at the same time.
[codecarbon INFO @ 15:40:40] Emissions data (if any) will be saved to file c:\Users\kykin\OneDrive\Desktop\DSAN5550\dsan_5550_project\data\carbon_logs\decision_tree_builder.csv
[codecarbon INFO @ 15:40:40] [setup] RAM Tracking...
[codecarbon INFO @ 15:40:40] [setup] CPU Tracking...
[codecarbon INFO @ 15:40:40] [setup] GPU Tracking...
[codecarbon INFO @ 15:40:40] Tracking CPU via the Windows Energy Meter Interface (RAPL)
[codecarbon INFO @ 15:40:40] Tracking Nvidia GPUs via PyNVML
[codecarbon INFO @ 15:40:41] The below tracking methods have been set up:
                RAM Tracking Method: RAM power estimation model
                CPU Tracking Method: Windows EMI
                GPU Tracking Method: pynvml
            
[co

In [ ]:
import pandas as pd
df = pd.read_csv('data/data_2015_2020/edited_dataset.csv')

In [12]:
# creating a sample to test the decision tree
sampled_df = df.sample(frac=0.2)

# Nodes
The first node on the tree will look at how well each car matches the flat budget. If the user is willing to buy a used car, this will look at the depreciated car values instead of the flat msrp.

In [105]:
cols = ['make_id','model_id','submodel_id','trim_id','year','make','model',
        'submodel','trim','description','msrp','depreciated_cost',
        'body_type','engine_type']

scorecard = sampled_df[cols].copy()

In [106]:
import math
import numpy as np

def budget_fit_score(price, budget,  k_under=50000, k_over=5000):
    """Best score at or near budget, slow decay if below, fast decay if over"""
    if price is None or budget is None or budget <= 0:
        return None
    k = k_under if price < budget else k_over
    return round(float(np.exp(-abs(price - budget) / k)),4)

In [107]:
for p in [20000, 21250, 25000, 30000, 35000, 40000, 50000]:
    print(p, round(budget_fit_score(p, 20000), 3))

20000 1.0
21250 0.779
25000 0.368
30000 0.135
35000 0.05
40000 0.018
50000 0.003


In [108]:
BUDGET = 20000
BUDGET_WEIGHT = 0.10

scorecard["budget_subscore"] = scorecard["depreciated_cost"].apply(lambda p: budget_fit_score(p, BUDGET))

print(scorecard[['year','make','model',"msrp","depreciated_cost","budget_subscore"]].sort_values('budget_subscore', ascending = False).head())

       year       make           model   msrp  depreciated_cost  \
3306   2019  Chevrolet         Equinox  23800          19992.00   
12518  2016   Chrysler  Town & Country  38445          19991.40   
17150  2015     Toyota         Sequoia  44395          19977.75   
5560   2019     Toyota           Prius  23770          19966.80   
13761  2016     Nissan            370Z  38370          19952.40   

       budget_subscore  
3306            0.9998  
12518           0.9998  
17150           0.9996  
5560            0.9993  
13761           0.9990  


In [109]:
def fuel_cost_score(gas_price_gal, kwh_price, avg_gal_gas_per_year, avg_kwh_per_year, x_min = 400, x_max = 5000):
    """low cost = high score, linear on a log scale"""
    yearly_gas = gas_price_gal*avg_gal_gas_per_year
    yearly_kwh = kwh_price*avg_kwh_per_year
    annual_cost = yearly_gas + yearly_kwh

    return float(
        (np.log(x_max) - np.log(annual_cost)) / (np.log(x_max) - np.log(x_min))
    )

In [110]:
GAS_PRICE = 4.10
KWH_PRICE = 0.18

scorecard["fuel_subscore"] = sampled_df.apply(
    lambda r: fuel_cost_score(
        GAS_PRICE,
        KWH_PRICE,
        r["avg_gal_gas_per_year"],
        r["avg_kwh_per_year"],
    ),
    axis=1,
)

In [111]:
print(scorecard[['year','make','model', 'engine_type',"msrp","depreciated_cost","budget_subscore", "fuel_subscore"]].sort_values('fuel_subscore', ascending = False).head())

      year     make        model     engine_type   msrp  depreciated_cost  \
2611  2020   Toyota  Prius Prime  plug-in hybrid  29500           29500.0   
4178  2019  Hyundai        Ioniq        electric  30315           25464.6   
4177  2019  Hyundai        Ioniq        electric  36815           30924.6   
7238  2018  Hyundai        Ioniq        electric  36000           25920.0   
4179  2019  Hyundai        Ioniq  plug-in hybrid  29350           24654.0   

      budget_subscore  fuel_subscore  
2611           0.1496       0.861842  
4178           0.3352       0.835252  
4177           0.1125       0.835252  
7238           0.3061       0.835252  
4179           0.3942       0.800973  


## Fuel savings
an alternative to a raw score would be a fuel savings score. This compares against an average car and an average cost of fuel each year, which likely means we are comparing to a mid-sized SUV

https://coltura.org/us-gasoline-consumption/
"According to the latest annual U.S. Bureau of Labor Statistics Consumer Expenditure data currently available, the average U.S. consumer unit spent $2,411 on gasoline in 2024". This is the most up-to-date figure

In [112]:
def fuel_savings_score(gas_price_gal, kwh_price, avg_yearly_cost,
                       avg_gal_gas_per_year, avg_kwh_per_year, decay_k=800):
    """Comparison to the yearly gas cost for an average american."""

    yearly_gas = gas_price_gal * avg_gal_gas_per_year
    yearly_kwh = kwh_price * avg_kwh_per_year
    annual_cost = yearly_gas + yearly_kwh

    if annual_cost <= 0.5 * avg_yearly_cost:
        return 1.0
    if annual_cost <= avg_yearly_cost:
        return 0.5 + 0.5 * (avg_yearly_cost - annual_cost) / (avg_yearly_cost - 0.5 * avg_yearly_cost)
    return 0.5 * float(np.exp(-(annual_cost - avg_yearly_cost) / decay_k))

In [113]:
GAS_PRICE = 4.10
KWH_PRICE = 0.18
AVG_YEARLY_COST = 2411

scorecard["fuel_savings_subscore"] = sampled_df.apply(
    lambda r: fuel_savings_score(
        GAS_PRICE,         
        KWH_PRICE,         
        AVG_YEARLY_COST,      
        r["avg_gal_gas_per_year"],
        r["avg_kwh_per_year"],
    ),
    axis=1,
)

In [114]:
print(scorecard[['year','make','model', 'engine_type',"msrp","depreciated_cost","budget_subscore", "fuel_subscore", "fuel_savings_subscore"]].sort_values('fuel_savings_subscore', ascending = False).head())

       year           make   model     engine_type    msrp  depreciated_cost  \
1603   2020            Kia    Niro  plug-in hybrid   36390          36390.00   
11712  2017     Volkswagen  e-Golf        electric   36995          22566.95   
4883   2019  Mercedes-Benz   S560e  plug-in hybrid  106400          89376.00   
2151   2020         Nissan    Leaf        electric   31600          31600.00   
5485   2019          Tesla       S        electric   94000          78960.00   

       budget_subscore  fuel_subscore  fuel_savings_subscore  
1603            0.0377       0.762893                    1.0  
11712           0.5985       0.790382                    1.0  
4883            0.0000       0.601388                    1.0  
2151            0.0983       0.763066                    1.0  
5485            0.0000       0.725331                    1.0  


In [115]:
def fuel_cost(gas_price_gal, kwh_price, avg_yearly_cost, avg_gal_gas_per_year, avg_kwh_per_year):
    """just doing math""" 
    yearly_gas = gas_price_gal*avg_gal_gas_per_year
    yearly_kwh = kwh_price*avg_kwh_per_year
    annual_cost = yearly_gas + yearly_kwh
    return annual_cost

In [116]:
GAS_PRICE = 3.20
KWH_PRICE = 0.14
AVG_YEARLY = 2144

scorecard["annual_fuel_cost"] = sampled_df.apply(
    lambda r: fuel_cost(
        GAS_PRICE, KWH_PRICE, AVG_YEARLY,
        r["avg_gal_gas_per_year"], r["avg_kwh_per_year"],
    ),
    axis=1,
)

In [117]:
print(scorecard[['year','make','model', 'engine_type',"msrp","depreciated_cost","budget_subscore", "fuel_subscore", "fuel_savings_subscore", "annual_fuel_cost"]].sort_values('fuel_savings_subscore', ascending = False).head())

       year           make   model     engine_type    msrp  depreciated_cost  \
1603   2020            Kia    Niro  plug-in hybrid   36390          36390.00   
11712  2017     Volkswagen  e-Golf        electric   36995          22566.95   
4883   2019  Mercedes-Benz   S560e  plug-in hybrid  106400          89376.00   
2151   2020         Nissan    Leaf        electric   31600          31600.00   
5485   2019          Tesla       S        electric   94000          78960.00   

       budget_subscore  fuel_subscore  fuel_savings_subscore  annual_fuel_cost  
1603            0.0377       0.762893                    1.0        567.665591  
11712           0.5985       0.790382                    1.0        528.259200  
4883            0.0000       0.601388                    1.0        853.785456  
2151            0.0983       0.763066                    1.0        565.992000  
5485            0.0000       0.725331                    1.0        622.591200  


In [118]:
emissions = tracker.stop()
print(f"{emissions:.6f} kg CO2eq")

[codecarbon INFO @ 16:58:22] Energy consumed for RAM : 0.025877 kWh. RAM Power : 20.0 W
[codecarbon INFO @ 16:58:22] Energy consumed for all GPUs : 0.013676 kWh. Total GPU Power : 13.321808352360586 W
[codecarbon INFO @ 16:58:22] Delta energy consumed for CPU with windows_emi : 0.000056 kWh, power : 30.217645598682143 W
[codecarbon INFO @ 16:58:22] Energy consumed for All CPU : 0.034297 kWh
[codecarbon INFO @ 16:58:22] 0.073850 kWh of electricity and 0.000000 L of water were used since the beginning.
[codecarbon INFO @ 16:58:22] 0.003451 g.CO2eq/s mean an estimation of 108.81582470183636 kg.CO2eq/year


0.018075 kg CO2eq
